# 🏗️ Construction Intelligence Hub
### End-to-End Machine Learning Pipeline (Regression & Classification)

This notebook automatically:
- Loads your uploaded CSV dataset
- Performs Exploratory Data Analysis (EDA)
- Cleans and preprocesses the data
- Detects whether the task is **Regression** or **Classification**
- Trains and compares multiple ML models
- Selects and saves the **best model** using Joblib
- Lets you make predictions on new data

> **Instructions:** Run the cells from top to bottom. When prompted, upload your CSV file (~50,000 rows, 10 columns). No code changes are required.

## 1️⃣ Import Libraries
All libraries required for data handling, visualization, preprocessing, modeling, and saving the model.

In [ ]:
# ----- Core Libraries -----
import pandas as pd
import numpy as np

# ----- Visualization -----
import matplotlib.pyplot as plt
import seaborn as sns

# ----- Preprocessing & Model Selection -----
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ----- Regression Models -----
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVC

# ----- Evaluation Metrics -----
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# ----- Saving the Model -----
import joblib

# ----- Warnings -----
import warnings
warnings.filterwarnings("ignore")

# ----- Plot Style -----
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✅ All libraries imported successfully.")

## 2️⃣ Upload Dataset
Upload your CSV file (approximately 50,000 rows × 10 columns) using the Colab file upload widget below.

In [ ]:
from google.colab import files

print("📤 Please upload your CSV dataset (approx. 50,000 rows, 10 columns)...")
uploaded = files.upload()

# Get the uploaded file name automatically
uploaded_filename = list(uploaded.keys())[0]
print(f"\n✅ File uploaded successfully: {uploaded_filename}")

## 3️⃣ Load Dataset
Load the uploaded CSV into a pandas DataFrame.

In [ ]:
# Load the CSV file into a DataFrame
df = pd.read_csv(uploaded_filename)

# Keep an untouched copy of the original data for reference
df_original = df.copy()

print(f"✅ Dataset loaded successfully with shape: {df.shape}")
df.head()

## 4️⃣ Exploratory Data Analysis (EDA)
Understand the structure, quality, and relationships within the dataset before modeling.

In [ ]:
# ----- First 5 Rows -----
print("🔹 First 5 Rows of the Dataset:")
df.head()

In [ ]:
# ----- Dataset Shape -----
print(f"🔹 Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
# ----- Dataset Info -----
print("🔹 Dataset Info:")
df.info()

In [ ]:
# ----- Data Types -----
print("🔹 Data Types of Each Column:")
df.dtypes

In [ ]:
# ----- Missing Values -----
print("🔹 Missing Values per Column:")
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_percent})
missing_df[missing_df["Missing Count"] > 0].sort_values("Missing Count", ascending=False)

In [ ]:
# ----- Duplicate Rows -----
duplicate_count = df.duplicated().sum()
print(f"🔹 Number of Duplicate Rows: {duplicate_count}")

In [ ]:
# ----- Summary Statistics -----
print("🔹 Summary Statistics (Numerical Columns):")
df.describe(include=[np.number])

In [ ]:
# ----- Summary Statistics (Categorical Columns) -----
print("🔹 Summary Statistics (Categorical Columns):")
df.describe(include=["object", "category"])

In [ ]:
# ----- Correlation Heatmap -----
numeric_df = df.select_dtypes(include=[np.number])

if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(10, 8))
    sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title("Correlation Heatmap of Numerical Features", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Not enough numerical columns to generate a correlation heatmap.")

## 5️⃣ Data Preprocessing
Steps performed:
1. Remove duplicate rows
2. Handle missing values (numeric → median, categorical → mode)
3. Encode categorical columns automatically using Label Encoding
4. Scale numerical features (excluding the target) using StandardScaler

In [ ]:
# ----- Remove Duplicate Rows -----
before_rows = df.shape[0]
df = df.drop_duplicates()
after_rows = df.shape[0]
print(f"✅ Removed {before_rows - after_rows} duplicate rows. New shape: {df.shape}")

In [ ]:
# ----- Handle Missing Values -----
# Numerical columns -> fill with median
# Categorical columns -> fill with mode (most frequent value)

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in [np.float64, np.int64]:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

print("✅ Missing values handled.")
print(f"Remaining missing values: {df.isnull().sum().sum()}")

## 6️⃣ Target Column Detection
The notebook attempts to automatically detect the target (label) column using common naming patterns
(e.g. `target`, `label`, `class`, `price`, `cost`, `outcome`, `result`, `status`, `type`).

If no suitable column can be identified automatically, you will be **asked to enter the target column name manually** in the input prompt below.

In [ ]:
# ----- Automatic Target Column Detection -----

# Common keywords that often indicate a target/label column
target_keywords = ["target", "label", "class", "output", "result", "outcome",
                    "price", "cost", "status", "type", "category", "risk",
                    "delay", "quality", "score", "rating", "failure", "defect"]

detected_target = None

# 1. Look for a column whose name contains one of the keywords
for col in df.columns:
    if any(keyword in col.lower() for keyword in target_keywords):
        detected_target = col
        break

# 2. If nothing matched, fall back to the LAST column
#    (a very common convention in ML datasets)
if detected_target is None:
    detected_target = df.columns[-1]
    print(f"⚠️ No column matched common target keywords. Defaulting to the last column: '{detected_target}'")
else:
    print(f"✅ Automatically detected target column: '{detected_target}'")

print("\nAvailable columns:", list(df.columns))

# ----- Manual Override -----
# If the automatically detected column is NOT correct, type the correct
# column name below and press Enter. Press Enter WITHOUT typing anything
# to keep the automatically detected column.
user_input = input(f"\nPress Enter to confirm '{detected_target}' as the target column, "
                    f"or type the correct column name: ").strip()

if user_input != "" and user_input in df.columns:
    target_column = user_input
    print(f"✅ Target column set manually to: '{target_column}'")
else:
    target_column = detected_target
    print(f"✅ Target column confirmed as: '{target_column}'")

## 7️⃣ Feature Engineering
- Separate features (X) and target (y)
- Determine whether this is a **Regression** or **Classification** problem
- Encode all categorical columns automatically using Label Encoding
- Scale numerical feature columns using StandardScaler

In [ ]:
# ----- Separate Features and Target -----
X = df.drop(columns=[target_column])
y = df[target_column]

# ----- Determine Problem Type -----
# If the target is numeric AND has many unique values -> Regression
# If the target is categorical OR has few unique numeric values -> Classification

unique_target_values = y.nunique()

if pd.api.types.is_numeric_dtype(y) and unique_target_values > 15:
    problem_type = "regression"
else:
    problem_type = "classification"

print(f"🔍 Detected problem type: {problem_type.upper()}")
print(f"🔍 Target column: '{target_column}' | Unique values: {unique_target_values}")

In [ ]:
# ----- Encode Categorical Feature Columns -----
label_encoders = {}
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print(f"✅ Encoded {len(categorical_cols)} categorical feature column(s): {categorical_cols}")

# ----- Encode Target Column (only if classification and non-numeric) -----
target_encoder = None
if problem_type == "classification" and not pd.api.types.is_numeric_dtype(y):
    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(y.astype(str))
    print(f"✅ Target column '{target_column}' encoded. Classes: {list(target_encoder.classes_)}")
elif problem_type == "classification":
    # Even numeric class labels are re-encoded to ensure they are 0-indexed and contiguous
    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(y)
    print(f"✅ Numeric target re-encoded for classification. Classes: {list(target_encoder.classes_)}")

In [ ]:
# ----- Feature Scaling -----
# Scale numerical features so that models like Logistic Regression and SVM
# (which are sensitive to feature magnitude) perform well.

scaler = StandardScaler()
numeric_feature_cols = X.select_dtypes(include=[np.number]).columns.tolist()

X_scaled = X.copy()
X_scaled[numeric_feature_cols] = scaler.fit_transform(X[numeric_feature_cols])

print(f"✅ Scaled {len(numeric_feature_cols)} numerical feature column(s).")
X_scaled.head()

## 8️⃣ Train/Test Split
Split the dataset into 80% training data and 20% testing data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42,
    stratify=y if problem_type == "classification" else None
)

print(f"✅ Training set shape: {X_train.shape}")
print(f"✅ Testing set shape:  {X_test.shape}")

## 9️⃣ Model Training
Depending on the detected problem type, the following models are trained:

**Regression:** Linear Regression, Decision Tree Regressor, Random Forest Regressor
**Classification:** Logistic Regression, Decision Tree Classifier, Random Forest Classifier, SVM

In [ ]:
# Dictionary to store trained models
trained_models = {}

if problem_type == "regression":
    models = {
        "Linear Regression": LinearRegression(),
        "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
        "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42)
    }
else:
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Decision Tree Classifier": DecisionTreeClassifier(random_state=42),
        "Random Forest Classifier": RandomForestClassifier(n_estimators=100, random_state=42),
        "Support Vector Machine (SVM)": SVC(kernel="rbf", probability=True, random_state=42)
    }

# ----- Train Each Model -----
for name, model in models.items():
    print(f"⏳ Training {name} ...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"✅ {name} trained successfully.\n")

print("🎉 All models trained successfully!")

## 🔟 Model Evaluation
Each model is evaluated using the appropriate metrics:

**Regression:** MAE, MSE, RMSE, R² Score
**Classification:** Accuracy, Precision, Recall, F1 Score, Confusion Matrix, Classification Report

In [ ]:
# Dictionary to store evaluation results and predictions
results = {}
predictions_store = {}

if problem_type == "regression":
    for name, model in trained_models.items():
        y_pred = model.predict(X_test)
        predictions_store[name] = y_pred

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        results[name] = {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2 Score": r2}

        print(f"📊 {name}")
        print(f"   MAE  : {mae:.4f}")
        print(f"   MSE  : {mse:.4f}")
        print(f"   RMSE : {rmse:.4f}")
        print(f"   R²   : {r2:.4f}")
        print("-" * 50)

else:
    for name, model in trained_models.items():
        y_pred = model.predict(X_test)
        predictions_store[name] = y_pred

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

        results[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1 Score": f1}

        print(f"📊 {name}")
        print(f"   Accuracy  : {acc:.4f}")
        print(f"   Precision : {prec:.4f}")
        print(f"   Recall    : {rec:.4f}")
        print(f"   F1 Score  : {f1:.4f}")
        print("\n   Classification Report:")
        print(classification_report(y_test, y_pred, zero_division=0))
        print("-" * 60)

In [ ]:
# ----- Confusion Matrices (Classification Only) -----
if problem_type == "classification":
    n_models = len(trained_models)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
    if n_models == 1:
        axes = [axes]

    for ax, (name, y_pred) in zip(axes, predictions_store.items()):
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
        ax.set_title(f"Confusion Matrix\n{name}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Confusion matrices are only generated for classification problems.")

In [ ]:
# ----- Actual vs Predicted Plots (Regression Only) -----
if problem_type == "regression":
    n_models = len(trained_models)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
    if n_models == 1:
        axes = [axes]

    for ax, (name, y_pred) in zip(axes, predictions_store.items()):
        ax.scatter(y_test, y_pred, alpha=0.4, edgecolor="k", s=15)
        min_val, max_val = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], "r--", lw=2)
        ax.set_title(f"Actual vs Predicted\n{name}")
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")

    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Actual vs Predicted plots are only generated for regression problems.")

## 1️⃣1️⃣ Model Comparison
Compare all trained models side-by-side in a single results table.

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(
    by="R2 Score" if problem_type == "regression" else "Accuracy",
    ascending=False
)

print("📋 Model Comparison Table:")
results_df

In [ ]:
# ----- Model Performance Comparison Chart -----
metric_to_plot = "R2 Score" if problem_type == "regression" else "Accuracy"

plt.figure(figsize=(10, 6))
bars = plt.bar(results_df.index, results_df[metric_to_plot], color=sns.color_palette("viridis", len(results_df)))
plt.title(f"Model Performance Comparison ({metric_to_plot})", fontsize=14)
plt.ylabel(metric_to_plot)
plt.xticks(rotation=20, ha="right")

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height, f"{height:.3f}",
              ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## 1️⃣2️⃣ Best Model Selection
The best model is automatically selected based on:
- **R² Score** (highest) for regression
- **Accuracy** (highest) for classification

In [ ]:
primary_metric = "R2 Score" if problem_type == "regression" else "Accuracy"
best_model_name = results_df[primary_metric].idxmax()
best_model = trained_models[best_model_name]

print(f"🏆 Best Performing Model: {best_model_name}")
print(f"🏆 {primary_metric}: {results_df.loc[best_model_name, primary_metric]:.4f}")

## 1️⃣3️⃣ Feature Importance
Feature importance is displayed for tree-based models (Decision Tree and Random Forest).

In [ ]:
tree_based_models = {name: model for name, model in trained_models.items()
                      if hasattr(model, "feature_importances_")}

if tree_based_models:
    fig, axes = plt.subplots(1, len(tree_based_models), figsize=(7 * len(tree_based_models), 5))
    if len(tree_based_models) == 1:
        axes = [axes]

    for ax, (name, model) in zip(axes, tree_based_models.items()):
        importances = pd.Series(model.feature_importances_, index=X_scaled.columns)
        importances = importances.sort_values(ascending=False)

        sns.barplot(x=importances.values, y=importances.index, ax=ax, palette="mako")
        ax.set_title(f"Feature Importance\n{name}")
        ax.set_xlabel("Importance")

    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ No tree-based models available for feature importance.")

## 1️⃣4️⃣ Save Best Model
Save the best-performing model, along with the scaler and encoders, using **Joblib**.
This ensures the exact same preprocessing is applied when making future predictions.

In [ ]:
# Bundle the model together with preprocessing objects so predictions
# can be made consistently on new, raw input data.
model_bundle = {
    "model": best_model,
    "model_name": best_model_name,
    "scaler": scaler,
    "label_encoders": label_encoders,
    "target_encoder": target_encoder,
    "feature_columns": list(X.columns),
    "numeric_feature_cols": numeric_feature_cols,
    "problem_type": problem_type,
    "target_column": target_column
}

joblib.dump(model_bundle, "best_model.pkl")
print("✅ Best model saved as 'best_model.pkl'")

# Download the file to your local machine (optional)
files.download("best_model.pkl")

## 1️⃣5️⃣ Prediction Example
Load the saved model and use it to make a prediction on a **new, unseen data sample**.

Replace the example values in `new_sample` below with your own input values.
The dictionary keys must match your original feature column names.

In [ ]:
# ----- Load the Saved Model Bundle -----
loaded_bundle = joblib.load("best_model.pkl")

loaded_model = loaded_bundle["model"]
loaded_scaler = loaded_bundle["scaler"]
loaded_encoders = loaded_bundle["label_encoders"]
loaded_target_encoder = loaded_bundle["target_encoder"]
loaded_feature_columns = loaded_bundle["feature_columns"]
loaded_numeric_cols = loaded_bundle["numeric_feature_cols"]
loaded_problem_type = loaded_bundle["problem_type"]

print(f"✅ Loaded model: {loaded_bundle['model_name']} ({loaded_problem_type})")
print(f"✅ Expected input features: {loaded_feature_columns}")

In [ ]:
# ----- Example: Build a New Sample Using the FIRST ROW of the original data -----
# 🔧 Replace these values with your own new data.
# The dictionary keys must exactly match the original feature column names.

new_sample = df_original[loaded_feature_columns].iloc[[0]].copy()
print("🔹 Example input row used for prediction:")
new_sample

In [ ]:
def predict_new_sample(sample_df, bundle):
    """
    Takes a raw single-row DataFrame (same columns as the original features)
    and returns the model's prediction, applying the same preprocessing
    (encoding + scaling) used during training.
    """
    sample_df = sample_df.copy()

    # Apply the same label encoders used during training
    for col, encoder in bundle["label_encoders"].items():
        if col in sample_df.columns:
            # Handle unseen categories gracefully
            sample_df[col] = sample_df[col].astype(str).apply(
                lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
            )

    # Apply the same scaler used during training
    sample_df[bundle["numeric_feature_cols"]] = bundle["scaler"].transform(
        sample_df[bundle["numeric_feature_cols"]]
    )

    # Ensure column order matches training data
    sample_df = sample_df[bundle["feature_columns"]]

    prediction = bundle["model"].predict(sample_df)

    # Decode classification target back to original labels, if applicable
    if bundle["problem_type"] == "classification" and bundle["target_encoder"] is not None:
        prediction = bundle["target_encoder"].inverse_transform(prediction)

    return prediction[0]


# ----- Make the Prediction -----
prediction_result = predict_new_sample(new_sample, loaded_bundle)
print(f"🎯 Predicted '{loaded_bundle['target_column']}': {prediction_result}")

## ✅ Summary

- Dataset uploaded, cleaned, and explored
- Problem type automatically detected (Regression or Classification)
- Multiple models trained and evaluated
- Best model selected automatically: see the **Best Model Selection** section above
- Best model saved as **`best_model.pkl`**
- Example prediction workflow provided for new data

You can now reuse `best_model.pkl` in any other notebook or application by loading it with `joblib.load("best_model.pkl")` and calling `predict_new_sample()`.